In [1]:
import sys, os
if 'google.colab' in sys.modules and not os.path.exists('.setup_complete'):
    # Install xvfb and our launcher script for it
    !apt-get install -y xvfb
    !wget -q https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/xvfb -O ../xvfb

    # Download dependencies from Github
    !wget https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/week06_policy_based/atari_wrappers.py
    !wget https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/week06_policy_based/env_batch.py
    !wget https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/week06_policy_based/runners.py

    # Update the gym environment to be compatible with the Atari environment
    !pip install -q gymnasium[atari,accept-rom-license]
    !pip install -q tensorboardX

    !touch .setup_complete

# This code creates a virtual display to draw game images on.
# It will have no effect if your machine has a monitor.
if type(os.environ.get("DISPLAY")) is not str or len(os.environ.get("DISPLAY")) == 0:
    !bash ../xvfb start
    os.environ['DISPLAY'] = ':1'

/bin/bash: ../xvfb: No such file or directory


# Implementing Advantage-Actor Critic (A2C)

In this notebook you will implement Advantage Actor Critic algorithm that trains on a batch of Atari 2600 environments running in parallel.

Firstly, we will use environment wrappers implemented in file `atari_wrappers.py`. These wrappers preprocess observations (resize, grayscale, take max between frames, skip frames and stack them together) and rewards. Some of the wrappers help to reset the environment and pass `done` flag equal to `True` when agent dies.
File `env_batch.py` includes implementation of `ParallelEnvBatch` class that allows to run multiple environments in parallel. To create an environment we can use `nature_dqn_env` function. Note that if you are using
PyTorch and not using `tensorboardX` you will need to implement a wrapper that will log **raw** total rewards that the *unwrapped* environment returns and redefine the implemention of `nature_dqn_env` function here.



In [2]:
import numpy as np
import gymnasium as gym
import ale_py
from atari_wrappers import nature_dqn_env

env_name = "SpaceInvadersNoFrameskip-v4"
nenvs = 4
summaries = "Tensorboard"

env = nature_dqn_env(env_name, nenvs=nenvs, summaries=summaries)
obs, _ = env.reset()
assert obs.shape == (nenvs, 4, 84, 84)
assert obs.dtype == np.float32

Next, we will need to implement a model that predicts logits and values. It is suggested that you use the same model as in [Nature DQN paper](https://www.nature.com/articles/nature14236) with a modification that instead of having a single output layer, it will have two output layers taking as input the output of the last hidden layer. **Note** that this model is different from the model you used in homework where you implemented DQN. You can use your favorite deep learning framework here. We suggest that you use orthogonal initialization with parameter $\sqrt{2}$ for kernels and initialize biases with zeros.

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class NatureCNN(nn.Module):
    def __init__(self, in_channels=4, n_actions=4):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 32, kernel_size=8, stride=4)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=4, stride=2)
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, stride=1)

        self.fc1 = nn.Linear(64 * 7 * 7, 512)
        self.policy_head = nn.Linear(512, n_actions)
        self.value_head = nn.Linear(512, 1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = x.reshape(x.size(0), -1)
        x = F.relu(self.fc1(x))
        logits = self.policy_head(x)
        value = self.value_head(x)
        return logits, value
def init_weights(m):
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)

You will also need to define and use a policy that wraps the model. While the model computes logits for all actions, the policy will sample actions and also compute their log probabilities.  `policy.act` should return a dictionary of all the arrays that are needed to interact with an environment and train the model.
 Note that actions must be an `np.ndarray` while the other
tensors need to have the type determined by your deep learning framework.

In [4]:
class Policy:
    def __init__(self, model):
        self.model = model
        self.device = next(self.model.parameters()).device

    def act(self, inputs):
        # Implement a policy by calling the model, sampling actions and computing their log probs.
        # Should return a dict containing keys ['actions', 'logits', 'log_probs', 'values'].
        x = torch.tensor(inputs, dtype=torch.float32, device=self.device)
        logits, values = self.model(x)
        probs = torch.softmax(logits, dim=-1)
        dist = torch.distributions.Categorical(probs)
        actions_t = dist.sample()
        log_probs = dist.log_prob(actions_t)

        return {
            'actions': actions_t.cpu().numpy(),
            'logits': logits,
            'log_probs': log_probs,
            'values': values.squeeze(-1)
        }


Next will pass the environment and policy to a runner that collects partial trajectories from the environment.
The class that does is is already implemented for you.

In [5]:
from runners import EnvRunner

This runner interacts with the environment for a given number of steps and returns a dictionary containing
keys

* 'observations'
* 'rewards'
* 'resets'
* 'actions'
* all other keys that you defined in `Policy`

under each of these keys there is a python `list` of interactions with the environment. This list has length $T$ that is size of partial trajectory. Partial trajectory for given moment `t` is part of `ComputeValueTargets.__call__` input argument `trajectory` from moment `t` to the end (i.e. it's different at each iteration in the algorithm).

To train the part of the model that predicts state values you will need to compute the value targets.
Any callable could be passed to `EnvRunner` to be applied to each partial trajectory after it is collected.
Thus, we can implement and use `ComputeValueTargets` callable.
The formula for the value targets is simple:

$$
\hat v(s_t) = \left( \sum_{t'=0}^{T - 1} \gamma^{t'}r_{t+t'} \right) + \gamma^T \hat{v}(s_{t+T}),
$$

In implementation, however, do not forget to use
`trajectory['resets']` flags to check if you need to add the value targets at the next step when
computing value targets for the current step. You can access `trajectory['state']['latest_observation']`
to get last observations in partial trajectory &mdash; $s_{t+T}$.

In [6]:
class ComputeValueTargets:
    def __init__(self, policy, gamma=0.99):
        self.policy = policy
        self.gamma = gamma

    def __call__(self, trajectory):
        """Compute value targets for a given partial trajectory."""

        # This method should modify trajectory inplace by adding
        # an item with key 'value_targets' to it.

        T = len(trajectory['rewards'])
        last_obs = trajectory['state']['latest_observation']
        device = self.policy.device

        obs_tensor = torch.tensor(last_obs, dtype=torch.float32, device=device)
        with torch.no_grad():
            _, values_last = self.policy.model(obs_tensor)
        next_value = values_last.squeeze(-1).cpu().numpy()

        value_targets = []
        for t in reversed(range(T)):
            r_t = trajectory['rewards'][t]
            reset_t = trajectory['resets'][t].astype(np.float32)
            not_done = 1.0 - reset_t
            target_t = r_t + self.gamma * not_done * next_value
            value_targets.insert(0, target_t)
            next_value = target_t

        trajectory['value_targets'] = value_targets

After computing value targets we will transform lists of interactions into tensors
with the first dimension `batch_size` which is equal to `env_steps * num_envs`, i.e. you essentially need
to flatten the first two dimensions.

In [7]:
class MergeTimeBatch:
    """ Merges first two axes typically representing time and env batch. """
    def __call__(self, trajectory):
        # Modify trajectory inplace.

        for key, value in list(trajectory.items()):
            if key == 'state' or not isinstance(value, list):
                continue
            first = value[0]
            if isinstance(first, torch.Tensor):
                trajectory[key] = torch.cat(value, dim=0)
            else:
                trajectory[key] = np.concatenate(value, axis=0)


In [8]:
model = NatureCNN(in_channels=4, n_actions=env.action_space.n)
model.apply(init_weights)
policy = Policy(model)
runner = EnvRunner(
    env=env,
    policy=policy,
    nsteps=5,
    transforms=[
        ComputeValueTargets(policy),
        MergeTimeBatch(),
    ],
)


Now is the time to implement the advantage actor critic algorithm itself. You can look into your lecture,
[Mnih et al. 2016](https://arxiv.org/abs/1602.01783) paper, and [lecture](https://www.youtube.com/watch?v=Tol_jw5hWnI&list=PLkFD6_40KJIxJMR-j5A1mkxK26gh_qg37&index=20) by Sergey Levine.

In [9]:
from torch.distributions import Categorical

class A2C:
    def __init__(self,
                 policy,
                 optimizer,
                 value_loss_coef=0.25,
                 entropy_coef=0.01,
                 max_grad_norm=0.5):
        self.policy = policy
        self.optimizer = optimizer
        self.value_loss_coef = value_loss_coef
        self.entropy_coef = entropy_coef
        self.max_grad_norm = max_grad_norm
        self.device = self.policy.device

    def _ensure_tensors(self, trajectory):
        if not isinstance(trajectory.get('value_targets'), torch.Tensor):
            trajectory['value_targets'] = torch.as_tensor(
                trajectory['value_targets'], dtype=torch.float32, device=self.device
            )

    def policy_loss(self, trajectory):
        log_probs = trajectory['log_probs']
        values = trajectory['values']
        value_targets = trajectory['value_targets']
        logits = trajectory['logits']

        advantage = value_targets - values.detach()
        pg_loss = -(log_probs * advantage).mean()

        dist = Categorical(logits=logits)
        entropy = dist.entropy().mean()

        return pg_loss - self.entropy_coef * entropy

    def value_loss(self, trajectory):
        values = trajectory['values']
        value_targets = trajectory['value_targets']
        return F.mse_loss(values, value_targets)

    def loss(self, trajectory):
        self._ensure_tensors(trajectory)
        p_loss = self.policy_loss(trajectory)
        v_loss = self.value_loss(trajectory)
        total_loss = p_loss + self.value_loss_coef * v_loss
        return total_loss, p_loss, v_loss

    def step(self, trajectory):
        self.optimizer.zero_grad()
        total_loss, p_loss, v_loss = self.loss(trajectory)
        total_loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(self.policy.model.parameters(), self.max_grad_norm)
        self.optimizer.step()

        # метрики для логирования
        with torch.no_grad():
            values = trajectory['values']
            value_targets = trajectory['value_targets']
            advantages = value_targets - values
            logits = trajectory['logits']
            dist = Categorical(logits=logits)
            entropy = dist.entropy().mean()

        self.last_metrics = {
            'advantages': advantages.detach().cpu().numpy(),
            'values': values.detach().cpu().numpy(),
            'value_targets': value_targets.detach().cpu().numpy(),
            'entropy': entropy.item(),
            'policy_loss': p_loss.item(),
            'value_loss': v_loss.item(),
            'total_loss': total_loss.item(),
            'grad_norm': grad_norm.item() if isinstance(grad_norm, torch.Tensor) else grad_norm,
        }

Now you can train your model. With reasonable hyperparameters training on a single GTX1080 for 10 million steps across all batched environments (which translates to about 5 hours of wall clock time)
it should be possible to achieve *average raw reward over last 100 episodes* (the average is taken over 100 last
episodes in each environment in the batch) of about 600. You should plot this quantity with respect to
`runner.step_var` &mdash; the number of interactions with all environments. It is highly
encouraged to also provide plots of the following quantities (these are useful for debugging as well):

* [Coefficient of Determination](https://en.wikipedia.org/wiki/Coefficient_of_determination) between
value targets and value predictions
* Entropy of the policy $\pi$
* Value loss
* Policy loss
* Value targets
* Value predictions
* Gradient norm
* Advantages
* A2C loss

For optimization we suggest you use RMSProp with learning rate starting from 7e-4 and linearly decayed to 0, smoothing constant (alpha in PyTorch and decay in TensorFlow) equal to 0.99 and epsilon equal to 1e-5.

In [10]:
%load_ext tensorboard
%tensorboard --logdir logs --port 61307

In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
# policy = Policy(model)   # теперь self.device будет 'cuda'

NatureCNN(
  (conv1): Conv2d(4, 32, kernel_size=(8, 8), stride=(4, 4))
  (conv2): Conv2d(32, 64, kernel_size=(4, 4), stride=(2, 2))
  (conv3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
  (fc1): Linear(in_features=3136, out_features=512, bias=True)
  (policy_head): Linear(in_features=512, out_features=6, bias=True)
  (value_head): Linear(in_features=512, out_features=1, bias=True)
)

In [12]:
import torch.optim as optim
from torch.optim.lr_scheduler import LinearLR
import numpy as np
import glob
import time
import os
from datetime import datetime
from atari_wrappers import TensorboardSummaries

save_dir = "checkpoints"
os.makedirs(save_dir, exist_ok=True)

start_step = 0
list_of_files = glob.glob(os.path.join(save_dir, "a2c_step*.pth"))


if list_of_files:
    def extract_step(filename):
        basename = os.path.basename(filename)
        parts = basename.split('_')
        try:
            step_str = parts[1]
            return int(step_str[4:])
        except:
            return 0

    latest_file = max(list_of_files, key=extract_step)
    checkpoint = torch.load(latest_file, map_location='cpu')

    if isinstance(checkpoint, dict) and 'step' in checkpoint:
        start_step = checkpoint['step']
        model.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint)
        start_step = extract_step(latest_file)

    print(f"Загружен чекпоинт {latest_file}, шаг = {start_step}")
else:
    print("Чекпоинтов не найдено, начинаем с нуля.")

tb_wrapper = env
while not isinstance(tb_wrapper, TensorboardSummaries):
    tb_wrapper = tb_wrapper.env
tb_wrapper.step_var = start_step


env.step_var = start_step

policy = Policy(model)

print("=== DEBUG ===")
print("start_step =", start_step)
print("env.step_var =", env.step_var)
runner = EnvRunner(
    env=env,
    policy=policy,
    nsteps=5,
    transforms=[ComputeValueTargets(policy, gamma=0.99), MergeTimeBatch()],
    step_var=start_step
)
print("runner.step_var =", runner.step_var)
print("============")

total_steps = 100_000_000
optimizer = optim.RMSprop(model.parameters(), lr=7e-4, alpha=0.99, eps=1e-5)
scheduler = LinearLR(optimizer, start_factor=1.0, end_factor=0.0, total_iters=total_steps)

a2c = A2C(policy, optimizer, value_loss_coef=0.5, entropy_coef=0.01, max_grad_norm=10.0) # entropy_coef=0.01

def compute_r2(values, targets):
    ss_res = np.sum((targets - values) ** 2)
    ss_tot = np.sum((targets - np.mean(targets)) ** 2)
    return 1 - ss_res / (ss_tot + 1e-8)

ep_advantages = []
ep_values = []
ep_targets = []
ep_entropies = []
ep_policy_losses = []
ep_value_losses = []
ep_total_losses = []
ep_grad_norms = []

iteration = 0
log_every = 100

try:
    while runner.step_var < total_steps:
        trajectory = runner.get_next()
        a2c.step(trajectory)
        scheduler.step()
    
        m = a2c.last_metrics
        ep_advantages.append(m['advantages'].mean())
        ep_values.append(m['values'].mean())
        ep_targets.append(m['value_targets'].mean())
        ep_entropies.append(m['entropy'])
        ep_policy_losses.append(m['policy_loss'])
        ep_value_losses.append(m['value_loss'])
        ep_total_losses.append(m['total_loss'])
        ep_grad_norms.append(m['grad_norm'])
    
        iteration += 1
    
        if iteration % log_every == 0:
            avg_adv = np.mean(ep_advantages)
            avg_val = np.mean(ep_values)
            avg_tgt = np.mean(ep_targets)
            avg_ent = np.mean(ep_entropies)
            avg_p_loss = np.mean(ep_policy_losses)
            avg_v_loss = np.mean(ep_value_losses)
            avg_loss = np.mean(ep_total_losses)
            avg_grad = np.mean(ep_grad_norms)
            r2 = compute_r2(m['values'], m['value_targets'])
    
            runner.add_summary('A2C/policy_loss', avg_p_loss)
            runner.add_summary('A2C/value_loss', avg_v_loss)
            runner.add_summary('A2C/entropy', avg_ent)
            runner.add_summary('A2C/advantage_mean', avg_adv)
            runner.add_summary('A2C/value_targets_mean', avg_tgt)
            runner.add_summary('A2C/value_predictions_mean', avg_val)
            runner.add_summary('A2C/grad_norm', avg_grad)
            runner.add_summary('A2C/total_loss', avg_loss)
            runner.add_summary('A2C/R2', r2)
    
            ep_advantages.clear()
            ep_values.clear()
            ep_targets.clear()
            ep_entropies.clear()
            ep_policy_losses.clear()
            ep_value_losses.clear()
            ep_total_losses.clear()
            ep_grad_norms.clear()

            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            fname = f"{save_dir}/a2c_step{runner.step_var}_{timestamp}.pth"
            torch.save({'step': runner.step_var, 'model_state_dict': model.state_dict()}, fname)
except KeyboardInterrupt:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    fname = f"{save_dir}/a2c_interrupt_{timestamp}.pth"
    torch.save({'step': runner.step_var, 'model_state_dict': model.state_dict()}, fname)
    print(f"\nОбучение прервано. Модель сохранена в {fname}")

torch.save({'step': runner.step_var, 'model_state_dict': model.state_dict()}, "a2c_spaceinvaders_final.pth")
print("Итоговая Модель сохранена.")
print("Обучение завершено!")

C:\Users\user\AppData\Local\Temp\ipykernel_11512\1793825560.py:28: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(latest_file, map_location='cpu')


Загружен чекпоинт checkpoints\a2c_step16562000_20260512_170117.pth, шаг = 16562000
=== DEBUG ===
start_step = 16562000
env.step_var = 16562000
runner.step_var = 16562000

Обучение прервано. Модель сохранена в checkpoints/a2c_interrupt_20260512_230430.pth
Итоговая Модель сохранена.
Обучение завершено!


# Тестируем, насколько модель научилась

In [15]:
import os
import glob
import torch
from atari_wrappers import nature_dqn_env

env_name = "SpaceInvadersNoFrameskip-v4"
nenvs = 1 
summaries = None

tmp_env = nature_dqn_env(env_name, nenvs=nenvs, summaries=summaries)
n_actions = tmp_env.action_space.n
tmp_env.close()

model = NatureCNN(in_channels=4, n_actions=n_actions)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

checkpoint_dir = "checkpoints"
pattern = os.path.join(checkpoint_dir, "a2c_step*.pth")
files = glob.glob(pattern)
if not files:
    pattern = "a2c_step*.pth"
    files = glob.glob(pattern)

def extract_step(filename):
    base = os.path.basename(filename)
    parts = base.split('_')
    for p in parts:
        if p.startswith("step"):
            return int(p[4:])
    return 0

if files:
    latest_file = max(files, key=extract_step)
    checkpoint = torch.load(latest_file, map_location='cpu')
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint)
    print(f"Загружен чекпоинт: {latest_file}")
else:
    raise FileNotFoundError("Не найдено ни одного чекпоинта a2c_step*.pth")

model.eval()
print("Модель успешно загружена и готова к игре.")

Загружен чекпоинт: checkpoints\a2c_step23572000_20260512_230427.pth
Модель успешно загружена и готова к игре.


C:\Users\user\AppData\Local\Temp\ipykernel_12812\2198713405.py:35: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(latest_file, map_location='cpu')


In [16]:
import numpy as np
import imageio

env = nature_dqn_env(env_name, nenvs=None, summaries=None, clip_reward=False)

frames = []
obs, _ = env.reset()
total_reward = 0
done = False
step = 0

print("Начинаем запись эпизода...")

while not done:
    frame = env.render()
    frames.append(frame)

    with torch.no_grad():
        obs_t = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
        logits, _ = model(obs_t)
        action = torch.argmax(logits, dim=-1).cpu().numpy()[0]

    obs, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated
    total_reward += reward
    step += 1

    if step % 100 == 0:
        print(f"Шаг {step}, награда за эпизод: {total_reward}")

print(f"Эпизод завершён. Общая награда: {total_reward}, шагов: {step}")
env.close()

video_path = "a2c_spaceinvaders_demo.mp4"
with imageio.get_writer(video_path, fps=30) as writer:
    for frame in frames:
        writer.append_data(frame)
print(f"Видео сохранено в {video_path}")

Начинаем запись эпизода...
Шаг 100, награда за эпизод: 55.0
Шаг 200, награда за эпизод: 155.0
Шаг 300, награда за эпизод: 355.0
Шаг 400, награда за эпизод: 410.0
Шаг 500, награда за эпизод: 425.0
Шаг 600, награда за эпизод: 490.0
Шаг 700, награда за эпизод: 535.0
Шаг 800, награда за эпизод: 560.0
Шаг 900, награда за эпизод: 620.0


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (160, 210) to (160, 224) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Шаг 1000, награда за эпизод: 795.0
Эпизод завершён. Общая награда: 795.0, шагов: 1008
Видео сохранено в a2c_spaceinvaders_demo.mp4


### Target networks?

You may recall a technique called "target networks" we used a few weeks ago when we trained a DQN agent to play Atari Breakout and wonder why we have not suggested using them here. The answer is that this is more historical than practical.

While the "chasing the target" problem is still present in actor-critic value estimation and target networks do show up in follow-up papers, the original A3C/A2C papers do not mention them and do not explain this omission.

The hypothesis why this may not be a big deal (compared to Q-learning) goes like this. An A3C/A2C agent selects actions based on policy, not an epsilon greedy exploration function, for which the argmax can change drastically due to tiny errors in function approximation. Therefore, errors in the value target caused by target chasing will cause less damage.

Also, the actor-critic gradient relies on the advantage function $A(s_t, a_t) = Q(s_t, a_t) - V(s_t)$. Compare this to the $Q$-function $Q(s_t, a_t) = r(s_t, a_t) + \gamma \cdot \mathbb{E}_{s_{t+1} \mid s_t, a_t} V(s_{t+1})$ used in Q-learning and SARSA: we would expect that any bias in $V$-function approximation will be carried over from $V(s_{t+1})$ to $V(s_t)$ by gradient updates. However, in the formula for the advantage function the two approximations ($Q$-function and $V$-function) come with opposite signs, and thus the errors will cancel out.

The last reason may be computational. Authors were concerned to beat existent algorithms in the wall-clock learning time, and any overhead of parameter copying (target network update) counted against this goal.